<a href="https://colab.research.google.com/github/furalmiqbadi/PembelajaranMesin_01_AbdulGhofurAlmiqbadi/blob/main/JS06/JS06_Tugas.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## 1.

In [31]:
# import librari
from pathlib import Path
import matplotlib.image as mpimg
import cv2
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.svm import SVC
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import accuracy_score

In [32]:
# load data

df = pd.read_csv('/content/voice.csv')

In [33]:
# pra-pengolahan data
X = df.drop('label', axis=1)
y = df['label']

In [34]:
# encoding label (male -> 0, female -> 1)
le = LabelEncoder()
y_encoded = le.fit_transform(y)

In [35]:
# scaling fitur (untuk SVM)
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

In [36]:
# eksperimen split dan kernel
results = []
splits = [(0.3, '70:30'), (0.2, '80:20')]
kernels = ['linear', 'poly', 'rbf']

for test_size, split_name in splits:
    # split data
    X_train, X_test, y_train, y_test = train_test_split(X_scaled, y_encoded, test_size=test_size, random_state=42)

    # latih model untuk setiap kernel
    for kernel in kernels:
        model = SVC(kernel=kernel, random_state=42)
        model.fit(X_train, y_train)

        # prediksi dan evaluasi
        y_pred = model.predict(X_test)
        acc = accuracy_score(y_test, y_pred)

        # simpan hasil
        results.append({
            'Rasio Split': split_name,
            'Kernel': kernel,
            'Akurasi': f"{acc:.4f}"
        })

In [37]:
# tabulasi hasil
df_results = pd.DataFrame(results)
print(df_results)

  Rasio Split  Kernel Akurasi
0       70:30  linear  0.9706
1       70:30    poly  0.9569
2       70:30     rbf  0.9811
3       80:20  linear  0.9763
4       80:20    poly  0.9685
5       80:20     rbf  0.9826


## 2.

In [38]:
import zipfile
import os
from pathlib import Path

In [39]:
# ekstrak file images.zip
zip_path = '/content/images.zip'
extract_path = '/content/'

with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_path)

In [40]:
# daftarkan path
train_dir = "images/training/"
test_dir = "images/test/"

In [45]:
# fungsi load_dataset
def load_dataset(img_dir):
    p = Path(img_dir)
    dirs = p.glob('*')
    img_list = []
    for dir in dirs:
        if dir.is_dir():
            label = str(dir).split('/')[-1]
            for file in list(dir.glob('*.jpg')) + list(dir.glob('*.png')):
                img = mpimg.imread(file)
                if img is not None:
                    img_list.append((img, label))
    return img_list

In [49]:
# load data training
train_img = load_dataset(train_dir)

In [50]:
# fungsi ekstraksi fitur histogram
def extract_histogram_features(img_list, target_size=(128, 128)):
    X = []
    y = []
    for img, label in img_list:
        # resize gambar
        img_resized = cv2.resize(img, target_size)

        # konversi ke grayscale
        img_gray = cv2.cvtColor(img_resized, cv2.COLOR_RGB2GRAY)

        # hitung histogram (256 bins)
        hist = cv2.calcHist([img_gray], [0], None, [256], [0, 256])
        # normalisasi histogram ke rentang 0 sampai 1 (untuk SVM RBF)
        hist = cv2.normalize(hist, hist, alpha=0, beta=1, norm_type=cv2.NORM_MINMAX)
        hist = hist.flatten()

        X.append(hist)
        # encode label day = 1, night = 0
        y.append(1 if label == 'day' else 0)

    return np.array(X), np.array(y)

X_hist, y_hist = extract_histogram_features(train_img)

In [51]:
# split data 80:20
X_train_img, X_test_img, y_train_img, y_test_img = train_test_split(
    X_hist, y_hist, test_size=0.2, random_state=42
)
print(f"Data Training: {X_train_img.shape[0]} gambar")
print(f"Data Testing: {X_test_img.shape[0]} gambar")

Data Training: 192 gambar
Data Testing: 48 gambar


In [52]:
# SVM kernel RBF (hyperparameter tuning)
param_grid = {
    'C': [0.1, 1, 10, 100],         # regularisasi
    'gamma': [1, 0.1, 0.01, 0.001]  # koefisien kernel RBF
}

svm_rbf = SVC(kernel='rbf', random_state=42)
# menggunakan 3-fold cv
grid_search = GridSearchCV(svm_rbf, param_grid, refit=True, verbose=2, cv=3)

grid_search.fit(X_train_img, y_train_img)

Fitting 3 folds for each of 16 candidates, totalling 48 fits
[CV] END .....................................C=0.1, gamma=1; total time=   0.0s
[CV] END .....................................C=0.1, gamma=1; total time=   0.0s
[CV] END .....................................C=0.1, gamma=1; total time=   0.0s
[CV] END ...................................C=0.1, gamma=0.1; total time=   0.0s
[CV] END ...................................C=0.1, gamma=0.1; total time=   0.0s
[CV] END ...................................C=0.1, gamma=0.1; total time=   0.0s
[CV] END ..................................C=0.1, gamma=0.01; total time=   0.0s
[CV] END ..................................C=0.1, gamma=0.01; total time=   0.0s
[CV] END ..................................C=0.1, gamma=0.01; total time=   0.0s
[CV] END .................................C=0.1, gamma=0.001; total time=   0.0s
[CV] END .................................C=0.1, gamma=0.001; total time=   0.0s
[CV] END .................................C=0.1,

GridSearchCV(cv=3, estimator=SVC(random_state=42),
             param_grid={'C': [0.1, 1, 10, 100],
                         'gamma': [1, 0.1, 0.01, 0.001]},
             verbose=2)

In [53]:
# evaluasi model terbaik
best_model = grid_search.best_estimator_
y_pred_img = best_model.predict(X_test_img)
accuracy_img = accuracy_score(y_test_img, y_pred_img)

print(f"Parameter Terbaik: {grid_search.best_params_}")
print(f"Akurasi pada Data Uji (Test Set): {accuracy_img:.4f}")

Parameter Terbaik: {'C': 10, 'gamma': 0.1}
Akurasi pada Data Uji (Test Set): 1.0000
